<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_103_output_layer_decoding_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 📊 **Module 103 — Output-Layer & Decoding Analysis** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)

> Phase 15 · Mechanistic Interpretability Workshop (Cohen ML4LLM deep-dives)


# Module 103 — Output-Layer & Decoding Analysis

> M27 / M70 / M89 ran evals at the *benchmark* level. Cohen Ch4 (projects 16-23) **opens the output head as data** — softmax distributions per token, sampling dynamics, prediction accuracy, loss/perplexity over text, and benchmark behaviour (HellaSwag, language-bias measurement) re-derived from first principles. This module makes you fluent in *every* number an LLM produces at inference — temperature, top-p, top-k, perplexity, surprisal, NLL, KL — and how each interacts with model quality, length normalization, and bias measurement.

### What you'll cover
1. The **output head** — logits → softmax → token distribution
2. **Temperature, top-k, top-p** — how each reshapes the distribution
3. **Probabilistic token selection** — sampling vs argmax, ancestral vs structured
4. **Token prediction accuracy** — top-1 / top-5 / token-by-token correctness
5. **The LLM loss function** — cross-entropy and per-token loss landscape
6. **Perplexity** — over time, over text, and the length-bias trap
7. **Position prediction probes** — linear vs logistic regression on hidden states
8. **HellaSwag and the multiple-choice eval** — how it actually works
9. **Measuring language bias** with conditional perplexity
10. The **2026 stack** — modern decoding tricks + production debugging


## 1 · The output head — what comes out of an LLM

```
final hidden state  h_t ∈ ℝ^D
       │
       ▼
     W_out · h_t   →   logits  z ∈ ℝ^V          # V ≈ 32k - 256k
       │
       ▼
     softmax(z / T)   →   probabilities  p ∈ [0,1]^V,  Σ p_i = 1
       │
       ▼
     argmax / sample   →   next token id  t̂
```

`W_out` is often **tied** to the input token embedding (M19/M20 / M102 callback) — same matrix used for input lookup *and* output projection. This is the **modern Transformer default** because it (a) halves the parameter count of the embedding/output layer, (b) regularizes the latent geometry, (c) keeps the input and output token spaces in agreement.

> 🧠 **Why decoupling matters for mech-interp.** Cohen Ch5 / M104 will use **logit lens** — projecting *intermediate* hidden states through `W_out` to read their "best-guess token" at every layer. This only works because `W_out` is shared with the embedding — the LLM is iteratively refining a distribution over the *same* vocabulary at every layer.


## 2 · Temperature, top-k, top-p

The four levers that shape the sampling distribution. Each reshapes `softmax(z / T)`:

| Lever | Effect | Default |
|---|---|---|
| **Temperature `T`** | `softmax(z / T)` — `T < 1` sharpens, `T > 1` flattens | `1.0` (raw), `0.7-1.0` chat, `0.0` greedy |
| **Top-k** | Keep only top `k` logits, renormalize | `k=40-50` chat, `k=1` greedy |
| **Top-p (nucleus, Holtzman 2019)** | Keep smallest set with `Σ p_i ≥ p` | `p=0.9-0.95` |
| **Min-p (Minh 2023)** | Keep `p_i ≥ min_p · max(p)` | `min_p=0.05-0.1` |
| **Repetition penalty** | Divide already-emitted token logits by `r > 1` | `r=1.0-1.3` |
| **Frequency / presence penalty** | OpenAI-style penalties on token count / occurrence | `0.0-2.0` |

**The decoding zoo of 2026:**

```
Greedy        →  pick argmax     →  one deterministic output (good for code, terrible for prose)
Beam search   →  track top-B sequences →  conservative summarization, captions (M95)
Temperature   →  random sample after T-scaling
Top-k         →  random sample after truncation
Top-p / Min-p →  random sample after nucleus
Mirostat      →  online temperature tuning to keep surprisal constant
Contrastive   →  penalize candidates similar to context (DoLa, contrastive decoding)
DRY / Tail-Free → modern anti-repetition + tail-cutting hybrids
Speculative   →  draft model + verify (M90)  — orthogonal speedup
```

**The "right" defaults in 2026:**
- **Chat (Claude, GPT-4o, Llama-3-Instruct)**: `T = 0.7-1.0`, `top_p = 0.9-0.95`, `min_p = 0.05`
- **Code / function-calling**: `T = 0.0` or `T = 0.2`, no top-p
- **Reasoning models (o1, R1, Claude-Thinking)**: `T = 0.0` for the reasoning trace, `T = 0.7` for the final answer
- **Creative writing / RP**: `T = 0.9-1.2`, min-p, DRY repetition penalty


In [ ]:
import torch, torch.nn.functional as F

def sample_with_levers(logits, temperature=1.0, top_k=0, top_p=0.0, min_p=0.0):
    z = logits / temperature
    if top_k > 0:
        v, _ = torch.topk(z, top_k)
        z[z < v[..., [-1]]] = -float("inf")
    p = F.softmax(z, dim=-1)
    if 0 < top_p < 1:
        sp, si = torch.sort(p, descending=True)
        cum = sp.cumsum(-1); mask = cum - sp > top_p
        sp[mask] = 0; p = torch.zeros_like(p).scatter_(-1, si, sp / sp.sum(-1, keepdim=True))
    if min_p > 0:
        p[p < min_p * p.max(-1, keepdim=True).values] = 0
        p = p / p.sum(-1, keepdim=True)
    return torch.multinomial(p, 1).squeeze(-1)


## 3 · Probabilistic token selection — ancestral, structured, repeat

Cohen proj 17 walks through the actual mechanics of **ancestral sampling**:

```
loop until EOS or max_length:
    h_t   = model(tokens)[:, -1, :]      # last-token hidden
    logits = W_out @ h_t                  # [V]
    p     = softmax(logits / T)
    next  = sample(p, top_k, top_p, min_p)
    tokens.append(next)
```

Two non-obvious behaviors:
1. **Sampling is NOT i.i.d. across time** — each new token reshapes context for the next. Small early divergences cascade.
2. **Determinism comes from the RNG seed + greedy decoding.** Setting `torch.manual_seed(42)` reproduces sampled outputs *only if* you also pin model dtype and check that no non-deterministic kernel was used (e.g. attention masking with NaNs).

**Structured decoding** (Outlines, lm-format-enforcer, guidance, Microsoft TypeChat, **constrained decoding in vLLM/SGLang/TensorRT-LLM**): at each step, **mask invalid tokens** based on a grammar (regex, JSON schema, CFG). The result *is guaranteed* to parse. The 2026 default for any agentic system emitting structured output.

> 🎯 **The 2026 lesson.** If your downstream code expects valid JSON / SQL / Python, **don't** post-process and hope. Use constrained decoding from the start. It's free (just masks logits), always-correct, and removes a huge class of LLM failures.


## 4 · Token prediction accuracy

Cohen proj 18: evaluate the model on a corpus by checking, at each position, whether the **true next token** is in the model's top-k predictions.

```
top-1 accuracy = fraction of positions where argmax(p) == y
top-5 accuracy = fraction where y ∈ topk(p, k=5)
top-50 accuracy = ...
```

Typical numbers on a held-out English corpus:
- GPT-2 (124M):  top-1 ≈ 36% · top-5 ≈ 58%
- Llama-3-8B:   top-1 ≈ 52% · top-5 ≈ 73%
- GPT-4-class:  top-1 ≈ 60% · top-5 ≈ 80%

**Two crucial caveats:**
1. **Frequent tokens dominate accuracy** — predicting "the" correctly is easy. Subset by token frequency to get a fair picture.
2. **Accuracy ≠ usefulness.** A model that gets a 60% top-1 *generation* still produces mostly-incorrect prose because errors compound.

The relationship between **top-1 accuracy** and **perplexity** is monotone but non-linear. Modern reports give both.


## 5 · The LLM loss — per-token cross-entropy

Cohen proj 19: the loss the model was *trained* on. For tokens `t_1, ..., t_N`:

```
L = (1/N) Σ_t  -log p_θ(t | t_<t)
  = (1/N) Σ_t  -log softmax(W_out · h_t)[t]
```

This is **per-token negative log-likelihood**, aka **NLL**, aka **cross-entropy**. The gradient of `L` wrt the model parameters is what backprop (M99) computes.

**Per-token-loss visualization** is the most powerful single mech-interp tool — plot `−log p_t` over the text and you'll see:
- **Spikes** at surprising or out-of-distribution tokens — proper nouns, rare numbers, idioms the model hasn't seen
- **Drops** at predictable continuations — "the cat sat on the ___" → "mat" is super low loss
- **Sentence boundaries** as small dips (period → capital letter is easy)

Spike analysis is how you diagnose *what your model doesn't know* without running a benchmark.


In [ ]:
# Cohen proj 19 — per-token loss over a passage
import torch, torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained("gpt2")
m   = AutoModelForCausalLM.from_pretrained("gpt2").eval()
text = "The Eiffel Tower is a wrought-iron lattice tower in Paris, France."
ids  = tok(text, return_tensors="pt").input_ids
with torch.no_grad():
    logits = m(ids).logits[:, :-1, :]      # predict next token
labels = ids[:, 1:]                         # shift
losses = F.cross_entropy(logits.reshape(-1, logits.size(-1)),
                         labels.reshape(-1), reduction="none")
for tk, l in zip([tok.decode([i]) for i in labels[0]], losses):
    print(f"{tk!r:>20}  −log p = {l.item():.2f}")


## 6 · Perplexity — over time, over text, and the length-bias trap

**Perplexity** = `exp(mean(−log p_t))`. It's a single scalar — the "average branching factor" of the language model.

```
PPL = exp(L)  where L = (1/N) Σ_t −log p_t
```

Cohen proj 20 visualizes perplexity in two ways:
1. **PPL over time during training** — should monotone-decrease (M99 LR schedule callback). Spikes = LR too high or bad data shard.
2. **PPL across documents** — Wikipedia, legal text, code, multilingual; reveals domain mismatch.

**Standard PPL numbers (2026, English Wikitext-103):**
- GPT-2 (124M):  ~30
- GPT-2 XL (1.5B): ~17
- Llama-3-8B: ~6.5
- Llama-3-70B: ~3.5
- GPT-4-class: ~3.0
- Theoretical floor (Shannon-style): ~1.5-2.0

**The length-bias trap.** PPL is sensitive to sequence length because:
- Longer contexts give the model more conditioning → lower PPL
- Some tokens (BOS, EOS) are easy to predict, dragging PPL down
- **Bytes-per-character (BPC)** normalizes by characters, fairer cross-tokenizer

Always compare PPL only across the **same tokenizer + same context length + same domain**, otherwise the comparison is meaningless.


## 7 · Linear / logistic regression on hidden states

Cohen proj 21 introduces the **probing classifier** — fit a *linear* model on hidden states `H = [h_t^{(l)}]_t` to predict some auxiliary signal:

| Probe target | Model | Tells you |
|---|---|---|
| **Token position** (regression) | Linear regression `pos ≈ w^T h_t^{(l)}` | Does this layer encode absolute position? |
| **Token POS tag** (classification) | Logistic regression | Where does syntactic info live? |
| **Sentiment of sentence** | Logistic regression on mean-pooled `H` | Is sentiment linearly readable? |
| **Truthfulness of statement** | Logistic regression (CCS, RepE) | Is honesty encoded as a direction? |

The seminal result: **"linear probes find structure that the model has learned to use."** If logistic regression on hidden states can predict POS tags with 95% accuracy, the model has internalized POS — no symbolic system needed.

**Caveat (Hewitt & Liang 2019).** Probes can find structure even when the model doesn't actually *use* it. Use **causal interventions** (M104 logit lens + activation patching) to confirm.

> 🪟 **Production application.** Linear probes are how Anthropic's "**activation engineering**" steers Claude — find the "refusal direction" via a logistic probe, *subtract* it from activations to reduce refusals. This is also how M89 Constitutional AI is hardened against jailbreaks at the activation level (rather than at the output level).


## 8 · HellaSwag and the multiple-choice eval

Cohen proj 22 dissects **HellaSwag** — one of the canonical LLM benchmarks. The format:

```
Context: A woman is outside with a bucket and a dog.
She is doing laundry by hand.
Then she ___
Choices: (A) puts the dog in the bucket   (B) walks back inside
         (C) gets the dog wet and dries it  (D) ties the dog up
```

**The model's job:** pick the most likely continuation. Mechanically:
```
For each choice c:
    score(c) = sum over tokens of c of log p_θ(token | context + earlier tokens of c)
return argmax_c score(c)
```

**The length-normalization issue.** Without normalization, shorter choices win because each token contributes negatively. Standard fix: **length-normalize by # tokens** in choice. Most LLM-eval-harness benchmarks (lm-eval-harness M70 callback) report both *length-normalized accuracy* and *raw accuracy*.

**Modern multiple-choice benchmarks (May 2026):**
- **MMLU / MMLU-Pro** — 57 academic subjects; standard knowledge eval
- **HellaSwag / Winogrande** — commonsense / coreference completion
- **ARC-Challenge / ARC-Easy** — grade-school science
- **AGIEval / BBH** — harder reasoning
- **GPQA Diamond** — graduate-level science (very hard)
- **MMLU-Redux / SimpleQA** — corrected / current-events versions

The Cohen-Ch4 lesson: **don't trust a single benchmark number.** Recompute the scoring, look at per-class accuracy, check the length distribution of correct vs incorrect answers. Every leaderboard top-spot has had at least one paper showing it can be gamed.


## 9 · Measuring language bias with conditional perplexity

Cohen proj 23 measures **bias** the most direct way: ask the model to complete a sentence and compute the per-token probability of biased vs anti-stereotypical continuations.

```
P("he is a programmer")  vs  P("she is a programmer")
P("the patient is a doctor")  vs  P("the patient is a nurse")
P("a Muslim man is")  vs  P("a Christian man is")    # toxicity probes
```

Cohen builds the simple test:
```
score = log p_θ(stereotypical) − log p_θ(anti-stereotypical)
```
Positive score = model prefers stereotype.

**Modern formalizations:**
- **StereoSet** (Nadeem 2021) — stereotype-anti-stereotype pairs across race, gender, religion, profession
- **CrowS-Pairs** (Nangia 2020) — 1508 minimally-different pairs
- **BBQ** (Bias Benchmark for QA, Parrish 2022) — ambiguous context + disambiguation
- **BOLD** (Dhamala 2021) — open-ended generation bias
- **HolisticBias** (Smith 2022) — 600 demographic descriptors × hundreds of prompts

**M100 callback.** These metrics produced the data behind every "model X is N% less biased than model Y" claim in 2024-2026 system cards. Llama-3 / Claude-3 / Gemini-2 all run them in pre-release evaluation.

> 📊 **What a bias score actually says.** A non-zero score means the *training data* and *post-training pipeline* together encoded that bias. Reducing it requires intervention at one of those stages — Constitutional AI (M89), targeted SFT on counterfactual examples, or post-hoc decoding interventions (like contrastive decoding away from the bias direction).


## 10 · 2026 decoding stack + production debugging

**Modern decoding tricks worth knowing:**

| Technique | What it does | When |
|---|---|---|
| **Speculative decoding** (M90) | Draft model proposes; target verifies | 2-4× speedup for any LLM |
| **DoLa / Contrastive decoding** (Li 2023) | Subtract early-layer logits from final | Reduces hallucinations |
| **Self-CD / Contrastive decoding** | Penalize amateur-model logits | Improves factuality |
| **CFG (Classifier-Free Guidance) for text** (Sanchez 2023) | `logits = (1+w)·conditional − w·unconditional` | Stronger instruction-following |
| **MBR decoding** (Eikema 2020) | Choose argmax_y E[utility(y, y')] over samples | Translation, summarization |
| **Repetition mitigation** (DRY, no-repeat-ngram, frequency penalty) | Various ways to penalize loops | Long-form generation |
| **Constrained / structured decoding** | Mask invalid tokens by grammar | JSON / SQL / function calling |
| **Length normalization** (`(score) / len^α`) | Fair comparison across choices | Multiple-choice eval, beam search |
| **Mirostat** | Online temperature to keep surprisal in target band | RP, creative writing |

**Production debug checklist** when generations are bad:
1. **Temperature too high or too low?** `T=0` for code, `T=0.7-1.0` for chat.
2. **Wrong chat template?** (M101 callback) — apply via `tokenizer.apply_chat_template`.
3. **Stop tokens correct?** Many models need `<|im_end|>` and `</s>` in `stop`.
4. **EOS not in vocab for fine-tuned model?** Check `tokenizer.eos_token_id` matches the actual emitted token.
5. **Constrained-decoding masks leaking valid tokens?** Test the schema separately.
6. **Repetition penalty too high?** > 1.3 starts breaking coherence.
7. **Top-p too low?** < 0.7 collapses creativity to greedy-like.

> 🎓 **The Cohen-Ch4 takeaway.** Every number an LLM produces — logit, probability, loss, perplexity, accuracy, bias score — is *interpretable*. The mech-interp craft is treating those numbers as data, plotting them, comparing across positions and across models, and using statistical / probing tools to pull out which patterns are real signal. After M101 + M102 + M103 you have the full toolkit for the M104-M106 *intervention* deep-dives.

## ✅ Recap

- **Output head** = `W_out·h_t → softmax(z/T) → token`; often tied to input embedding (enables logit lens, M104).
- **Decoding levers**: `temperature`, `top-k`, `top-p`, `min-p`, repetition penalty — each reshapes the distribution differently.
- **Constrained / structured decoding** (Outlines, lm-format-enforcer, vLLM JSON mode) is free and the 2026 default for agents.
- **Top-1 token accuracy** ≈ 36% (GPT-2) → 60% (GPT-4); frequent tokens dominate; subset by frequency.
- **Per-token loss / NLL** is the most powerful mech-interp tool — spikes reveal what the model doesn't know.
- **Perplexity** is comparable only *within* same tokenizer + length + domain; use BPC for cross-tokenizer.
- **Linear / logistic probes** find latent structure (POS, sentiment, truthfulness, refusal directions); always confirm with M104 interventions.
- **HellaSwag-style MC eval**: length-normalized log-prob over choice; report both raw and normalized.
- **Bias measurement** via stereotype/anti-stereotype perplexity gap (StereoSet, CrowS-Pairs, BBQ, BOLD, HolisticBias) — M100 production-required.
- **2026 stack**: temperature + top-p + min-p + structured decoding + speculative + DoLa + CFG-text + length-normalized scoring.

Next: **M104 — Residual-Stream Interventions** (Cohen Ch5).
